# Hera Exploration Notebook

This notebook demonstrates how to initialize a Hera **Project**, browse the **ToolkitHome** registry, and load data through a concrete toolkit (`MeteoLowFreq`).

## Prerequisites

- A running MongoDB instance
- The `hera` package installed (`pip install -e .` from the repo root)
- A project with data loaded (e.g., via `hera-toolkit repository load ...`)

## 1. Initialize a Project

The `Project` class is the central data access layer. It connects to MongoDB and provides CRUD operations for measurements, simulations, and cache documents.

In [ ]:
from hera import Project

# Replace with your actual project name
PROJECT_NAME = "MY_PROJECT"

proj = Project(projectName=PROJECT_NAME)

print(f"Project Name:     {proj.projectName}")
print(f"Files Directory:  {proj.filesDirectory}")
print(f"Measurements:     {type(proj.measurements).__name__}")
print(f"Simulations:      {type(proj.simulations).__name__}")
print(f"Cache:            {type(proj.cache).__name__}")

## 2. Browse the ToolkitHome Registry

`ToolkitHome` is the central registry of all available toolkits (built-in and dynamically registered). The singleton instance `toolkitHome` is created at import time.

In [ ]:
from hera import toolkitHome

# List all available toolkits (static + dynamic for this project)
toolkit_table = toolkitHome.getToolkitTable(projectName=PROJECT_NAME)
toolkit_table

## 3. Instantiate a MeteoLowFreq Toolkit

Use `toolkitHome.getToolkit()` to resolve, import, and instantiate a toolkit by name. The toolkit is connected to the project and can access its datasources.

In [ ]:
# Instantiate the MeteoLowFreq toolkit
lf_toolkit = toolkitHome.getToolkit("MeteoLowFreq", projectName=PROJECT_NAME)

print(f"Toolkit Name:  {lf_toolkit.toolkitName}")
print(f"Project Name:  {lf_toolkit.projectName}")
print(f"Analysis:      {lf_toolkit.analysis}")
print(f"Presentation:  {lf_toolkit.presentation}")

## 4. Explore Datasources

Each toolkit manages versioned datasources. Use `getDataSourceTable()` to see all datasources registered for this toolkit in the current project.

In [ ]:
# List all datasources for the MeteoLowFreq toolkit
ds_table = lf_toolkit.getDataSourceTable()
print("Datasource List:", lf_toolkit.getDataSourceList())
ds_table

## 5. Load Data via getDataSourceData

The `getDataSourceData()` method locates the datasource document in MongoDB, reads the `resource` path and `dataFormat`, and returns the actual data using the appropriate handler.

For parquet files, this returns a **dask DataFrame**. Call `.compute()` to materialize it into a pandas DataFrame.

In [ ]:
import pandas as pd

# Load data from the YAVNEEL datasource
# Replace 'YAVNEEL' with an actual datasource name from step 4
DATASOURCE_NAME = "YAVNEEL"

raw_data = lf_toolkit.getDataSourceData(DATASOURCE_NAME)

# Dask -> Pandas
if hasattr(raw_data, 'compute'):
    df = raw_data.compute()
else:
    df = raw_data

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 6. Use the Analysis Layer

The toolkit's `analysis` object provides domain-specific data processing methods.

In [ ]:
# Enrich the DataFrame with date-related columns
if 'datetime' in df.columns:
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
    enriched = lf_toolkit.analysis.addDatesColumns(df, datecolumn='datetime')
    print(f"New columns added: {set(enriched.columns) - set(df.columns)}")
    enriched.head()
else:
    print("No 'datetime' column found - check your datasource")

## 7. Use the Presentation Layer

The toolkit's `presentation` object provides plotting methods.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt

# Plot a scatter of Relative Humidity (RH) by hour of day
if 'RH' in df.columns:
    ax = lf_toolkit.presentation.dailyPlots.plotScatter(df, plotField='RH')
    plt.title('Relative Humidity - Daily Scatter')
    plt.tight_layout()
    plt.show()
else:
    print("No 'RH' column found - check your datasource")